# 🏦 개인실습 — 한빛은행 시맨틱 검색 (pgvector)

오늘 오전 시연(PostgreSQL 임베딩 DB 장점 & pgvector 기초)에서는 강사와 함께 **도서관 안내 문서 6개**를 pgvector로 검색해 봤죠 —
문서를 임베딩해 `documents`에 적재하고, `<=>`(코사인 거리)로 **의미가 가까운 문서**를 찾고,
`WHERE category=...`로 좁힌 뒤 벡터로 순위를 매겼어요.

이번에는 **여러분이 직접 코드를 처음부터 작성해서**, **한빛은행 FAQ/약관 검색기**를 만듭니다 —
예금·카드·대출·약관 등 **8개 문서**를 임베딩해 적재하고, "카드를 잃어버렸어요" 같은 자연어 질문으로
의미가 가까운 문서를 찾고, **⭐`WHERE category='카드'` + 벡터 검색을 하나의 SQL로 결합**해 좁혀 검색해요.

> 오늘의 핵심은 "명령을 외우는 것"이 아니라 **"임베딩 적재와 ⭐필터+벡터 결합을 내 손으로"** 예요.
> 도구(`psycopg`·`pgvector`·`register_vector`·`<=>`·BGE-M3·스키마)는 시연과 **똑같습니다**. **문서·카테고리·질의만 한빛은행으로 바뀝니다.**
> 시연 코드를 위로 스크롤해 베끼면 안 돼요 — 도메인이 다르거든요. 시연에서 '배운 절차'를 한빛은행으로 **옮겨오는** 연습입니다.

## 📋 이 실습에서 작성할 것 — 3개 실습(+ 🚀 도전 1개는 선택)
| 그룹 | 실습 | 다루는 것 |
|------|------|------|
| **A. 확장·스키마·모델** | 실습 1 | 벡터 확장 켜기 · `vector(1024)` 컬럼 · `encode`로 임베딩 |
| **B. 적재·시맨틱 검색** | 실습 2 | numpy 변환 · `<=>`(코사인 거리)로 검색 |
| **C. ⭐필터+벡터 결합** | 실습 3 | `WHERE` 카테고리 필터 + 벡터 검색 결합 |
| 🚀 도전(선택) | 실습 4 | 예금 카테고리로 자유적금 찾기 |

## ⏱️ 예상 시간표 (실습 82분 + 회고 18분 = 100분)
| 단계 | 실습 | 예상 |
|------|------|------|
| 준비 | 설치 + 오프라인 env·.env 접속·확장·문서 초기화·8문서·BGE-M3 (실행만) | 5분 |
| 실습 1 | [그룹 A] 확장 켜기·테이블·임베딩 맛보기 — 연계: 지난 시간 시연 pgvector 시작·미니 시맨틱 검색 | 20분 |
| 실습 2 | [그룹 B] 8문서 적재·시맨틱 검색(질의 1·2) — 지난 시간 시연 미니 시맨틱 검색 | 30분 |
| 실습 3 | [그룹 C] ⭐필터+벡터 결합(질의 3) — 지난 시간 시연 메타데이터 필터 + 벡터 결합 | 17분 |
| 실습 4 | 🚀 (도전) 자유적금 찾기(질의 4) — 지난 시간 시연 메타데이터 필터 + 벡터 결합 | 10분 |
| 회고 | 돌아보기 질문 | 18분 |

## ⚠️ 꼭 기억하세요
- 어제까지 쓰던 **`db-pg` 컨테이너(PostgreSQL 17 + pgvector)** 가 켜져 있어야 해요 (포트 5432). 기존 `customers`·`accounts`·`transactions` 은행 테이블은 **건드리지 않고**, `documents` 테이블만 새로 추가해요.
- **PostgreSQL은 비밀번호가 필요해요** — `.env` 파일에서 읽어요 (어제 Redis 무비밀번호와 **다른 점!**). 하드코딩하지 마세요.
- **오프라인 env를 모델 로딩보다 먼저!** `HF_HUB_OFFLINE`·`TRANSFORMERS_OFFLINE`를 `SentenceTransformer` 로딩 **전에** 설정해요 (안 하면 100초+ 지연).
- **`register_vector(conn)`은 확장을 켠 뒤(`CREATE EXTENSION vector`) 호출**해요 — numpy 배열을 `vector`로 바인딩해 주거든요.
- **`<=>`는 코사인 '거리'** 라 **작을수록 가까움** → `ORDER BY embedding <=> 질문` 은 오름차순(가장 가까운 게 위)이고, **`1 - (거리)` = 코사인 '유사도'** (1에 가까울수록 비슷).
- **채점은 "순위"로!** BGE-M3는 결정적 모델이라 **어느 문서가 1등인가(top-1)** 는 확정값이에요. 다만 **유사도 소수 수치는 근사(약 0.70·±0.01)** — 소수 뒷자리가 조금 달라도 **1등 문서만 아래 확정값과 같으면 성공**이에요. (순위가 다르면 오프라인 env·모델 캐시를 확인!)
- **검색까지가 오늘 범위** — 찾은 문서로 LLM이 답을 '생성'하는 건 rag-course에서 배워요. (오늘은 임베딩·검색만, LLM 미사용)
- 막히면 각 실습 아래 **💡 힌트**를 참고하세요. **정답은 별도 파일** `pgvector_3_한빛은행_시맨틱_검색_개인실습_정답.ipynb`에 있습니다.

---
## ⚙️ 준비 — 설치 + 오프라인 env·접속·문서 초기화·8문서·BGE-M3 (실행만 하세요)

아래 두 셀을 순서대로 실행하면 준비가 끝나요.
- 임베딩·pgvector 드라이버를 설치합니다.
- **오프라인 env를 먼저** 설정하고 → `.env`에서 `db-pg`(hanbit_bank)에 접속 → 확장을 켜고 `register_vector` → `documents`를 초기화(멱등) → 한빛은행 8문서를 정의 → **BGE-M3**를 로딩합니다.

> 이 준비 셀의 확장/접속은 **초기화용(완성)** 이에요. 실습 1에서 여러분이 **직접** 확장을 켜고 테이블을 만들어 봅니다 (똑같은 코드지만 손에 익혀요).
> 자동 채점 환경에서는 `PGPASSWORD` 환경변수가 있으면 그 값을 쓰고, 없으면 `.env`에서 물어봐요 (하드코딩 금지).

In [1]:
# 설치
# 이번 실습에 필요한 4개 패키지를 설치합니다. (-q: 설치 로그를 조용히 — quiet)
#   - sentence-transformers : 문장을 숫자 벡터(임베딩)로 바꿔주는 BGE-M3 모델을 불러올 때 사용
#   - pgvector              : PostgreSQL의 vector 타입을 파이썬 numpy 배열과 주고받게 해주는 어댑터
#   - "psycopg[binary]"     : 파이썬에서 PostgreSQL에 접속·SQL 실행을 담당하는 드라이버(라이브러리)
#   - python-dotenv         : .env 파일에 적어둔 비밀번호 등을 안전하게 읽어오는 도구
%pip install -q sentence-transformers pgvector "psycopg[binary]" python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 멱등 준비 — 오프라인 env → 접속 → 확장·register_vector → documents 초기화 → 8문서·BGE-M3
# ※ "멱등(idempotent)"이란 이 셀을 몇 번 다시 실행해도 결과가 항상 같다는 뜻입니다.
#    (예: DROP TABLE IF EXISTS로 먼저 지우고 다시 만들기 때문에, 두 번 실행해도 에러 없이 깨끗한 상태로 시작합니다.)
import os

# 📌 오프라인 env를 모델 로딩보다 먼저! (안 하면 100초+ 지연 — rag-course 실증)
# HF_HUB_OFFLINE, TRANSFORMERS_OFFLINE 두 환경변수를 "1"로 켜두면, sentence-transformers가
# 모델을 불러올 때 인터넷에서 새 버전이 있는지 확인하러 가지 않고 곧바로 로컬 캐시(내 컴퓨터에 이미
# 받아둔 파일)를 사용합니다. 이 줄을 model 관련 import보다 반드시 "먼저" 실행해야 효과가 있습니다.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

from dotenv import load_dotenv               # .env 파일 내용을 환경변수로 불러오는 함수
import numpy as np                           # 벡터(숫자 배열) 연산에 사용
import psycopg                               # PostgreSQL 접속·SQL 실행 드라이버
from pgvector.psycopg import register_vector # numpy 배열 ↔ PostgreSQL vector 타입을 자동 변환해주는 등록 함수
from sentence_transformers import SentenceTransformer  # 문장 → 벡터로 바꿔주는 임베딩 모델 클래스

# db-pg(hanbit_bank) 접속 — PostgreSQL은 비밀번호! (Redis와 다름) .env의 PGPASSWORD(load_dotenv, 하드코딩 금지)
load_dotenv()                                   # 같은 폴더의 .env → 환경변수
# .env 파일에는 보통 PGPASSWORD=이렇게생긴비밀번호 한 줄이 들어 있습니다.
# load_dotenv()를 실행하면 이 내용이 os.environ(환경변수 저장소)에 그대로 등록됩니다.
pw = os.environ["PGPASSWORD"]                   # 비밀번호는 .env 파일에 (하드코딩 금지)
# psycopg.connect(...)는 지정한 host(주소)·port(포트 번호)·dbname(데이터베이스 이름)으로
# PostgreSQL 서버에 접속을 시도하고, 성공하면 이후 SQL을 실행할 수 있는 conn(연결) 객체를 돌려줍니다.
conn = psycopg.connect(host="localhost", port=5432, dbname="hanbit_bank", user="postgres", password=pw)

# 벡터 확장 켜고(초기화용) register_vector 활성화 — 실습 1에서 직접 다시 켜봅니다
# CREATE EXTENSION IF NOT EXISTS vector : PostgreSQL에 pgvector 확장 기능(벡터 타입·거리 연산자)을
#   설치합니다. "IF NOT EXISTS"가 있어서 이미 설치돼 있으면 에러 없이 그냥 넘어갑니다.
conn.execute("CREATE EXTENSION IF NOT EXISTS vector")
conn.commit()                                   # commit()을 호출해야 변경 사항이 실제로 저장(확정)됩니다.
register_vector(conn)   # 📌 확장을 켠 뒤 호출 — numpy 배열을 vector로 바인딩
# register_vector(conn)을 해두면, 이후 이 conn으로 SQL을 실행할 때 numpy 배열을 자동으로
# PostgreSQL의 vector 타입으로 바꿔주고, 반대로 vector 컬럼을 읽어올 때도 numpy 배열로 돌려받습니다.

# 재실행 안전(멱등): documents만 비웁니다 (기존 customers/accounts/transactions 은행 테이블은 불간섭)
# DROP TABLE IF EXISTS documents : documents 테이블이 있으면 통째로 삭제합니다. ("IF EXISTS"라서
#   처음 실행이라 테이블이 아직 없어도 에러가 나지 않습니다.) 이 테이블은 실습 1에서 다시 만들 것입니다.
conn.execute("DROP TABLE IF EXISTS documents")
conn.commit()

# 한빛은행 FAQ/약관 8문서 (적재 순서대로 id 1~8 자동 부여)
# 아래 DOCS는 (문서 내용, 카테고리) 쌍 8개로 이루어진 파이썬 리스트입니다.
# 실습 1~4에서 이 문서들을 documents 테이블에 넣고, 문장을 벡터로 바꿔 유사도 검색을 연습합니다.
# id는 코드에 직접 적지 않고, 테이블에 넣는 "순서대로" BIGSERIAL(자동 증가)이 1부터 매겨집니다.
DOCS = [
    ("정기예금 금리는 연 3.5% 수준이며 상품별로 다르게 적용됩니다.", "예금"),         # id 1
    ("자유적금은 매달 자유롭게 납입할 수 있고 만기 시 우대 금리를 받습니다.", "예금"),   # id 2
    ("체크카드를 분실한 경우 고객센터나 모바일 앱에서 즉시 재발급을 신청할 수 있습니다.", "카드"),  # id 3
    ("신용카드 이용 한도는 고객 등급과 결제 실적에 따라 조정됩니다.", "카드"),         # id 4
    ("인터넷뱅킹 비밀번호를 5회 잘못 입력하면 계정이 잠깁니다.", "인터넷뱅킹"),        # id 5
    ("주택담보대출 한도는 소득과 담보 가치에 따라 결정됩니다.", "대출"),             # id 6
    ("예금자 보호법에 따라 원금과 이자를 합쳐 최대 5천만 원까지 보호됩니다.", "약관"),  # id 7
    ("타행 이체 수수료는 건당 500원이며 우수 등급 고객은 면제됩니다.", "이체"),        # id 8
]

# BGE-M3 임베딩 모델 (로컬 캐시·1024차원) — 로딩 약 5~10초(예시·기기마다 다름)
# SentenceTransformer("BAAI/bge-m3")는 문장을 1024개의 숫자로 이루어진 벡터로 바꿔주는 모델을
# 불러옵니다. 위에서 오프라인 env를 켜뒀기 때문에 인터넷 확인 없이 로컬 캐시에서 바로 불러옵니다.
model = SentenceTransformer("BAAI/bge-m3")

print("준비 완료 — db-pg(hanbit_bank) 접속·documents 초기화·8문서 정의·BGE-M3 로딩.")
print("문서 수:", len(DOCS))

c:\Users\Admin\Desktop\실습용\my_llm_service\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 35132.99it/s]


준비 완료 — db-pg(hanbit_bank) 접속·documents 초기화·8문서 정의·BGE-M3 로딩.
문서 수: 8


---
## 🧪 실습 1. [그룹 A] 벡터 확장 켜기 · 테이블 만들기 · 임베딩 맛보기 (예상 20분, 연계: 지난 시간 시연 pgvector 시작·미니 시맨틱 검색)

**시나리오**: 한빛은행 검색 개발자가 된 여러분의 첫 걸음 — `db-pg`를 **벡터 DB로** 만들고, 문서를 담을 **테이블**을 준비합니다.
시연에서 도서관 문서를 담을 때 했던 그대로, ① 확장을 켜고 ② `documents` 테이블을 만들고 ③ 문장 하나를 임베딩해 봅니다.

**요구사항**: ① PostgreSQL을 벡터 DB로 만드는 **확장**을 켜고 ② 1024개 숫자(임베딩)를 담는 **벡터 열**이 있는 `documents` 테이블을 만든 뒤 ③ `SentenceTransformer`로 문장 하나를 **벡터로 바꿔** 차원을 확인하는 코드를 작성하세요.

**✅ 확인 포인트**:
- `pgvector 확장 버전: 0.8.5` — 확장이 켜졌어요 ("PostgreSQL에 한 줄이면 벡터 DB")
- `documents 테이블 생성 완료`
- `첫 문서 임베딩 차원: (1024,)` — BGE-M3는 문장을 **1024개 숫자**로 바꿔요 (`vector(1024)`와 딱 맞음)

In [3]:
# 🧪 실습 1 — pgvector 확장 켜기 
# ✅ 포인트: PostgreSQL이 '벡터 DB'가 되는 한 줄 — CREATE EXTENSION vector (확장 0.8.5 내장·멱등).
from pgvector.psycopg import register_vector   # numpy 배열 ↔ PostgreSQL vector 타입을 자동 변환해주는 등록 함수

conn.execute("CREATE EXTENSION IF NOT EXISTS vector")   # 벡터 확장 활성화(library DB에 한 번)
conn.commit()                                            # DDL도 커밋(autocommit=False가 기본)

# ✅ 포인트: register_vector는 '확장 활성화 후'에 호출 — numpy 배열을 vector 파라미터로 바인딩해줍니다.
register_vector(conn)

# 확장 버전 확인 → 0.8.5 (이미지에 내장·이미지 교체나 재-pull 불필요)
# pg_extension은 설치된 확장 목록을 담은 시스템 테이블. WHERE extname='vector'로 그중 한 줄만 골라 버전 확인.
ver = conn.execute("SELECT extversion FROM pg_extension WHERE extname='vector'").fetchone()[0]
print("pgvector 확장 버전 →", ver)

pgvector 확장 버전 → 0.8.6


In [4]:
#documents 테이블 생성(1024차원 벡터 열) → 문장 하나를 임베딩
conn.execute("DROP TABLE IF EXISTS documents")
conn.execute("""
CREATE TABLE documents (
    id        BIGSERIAL PRIMARY KEY,   -- 문서 번호(자동 증가)
    content   TEXT,                    -- 문서 원문
    embedding vector(1024),            -- ✅ 의미의 좌표(BGE-M3 1024차원)
    category  TEXT                     -- 메타데이터(필터용 카테고리)
)
""")
conn.commit()
print("documents 테이블 생성 완료")

documents 테이블 생성 완료


In [5]:
# 📌 오프라인 env를 모델 로딩보다 먼저! (미설정 시 100초+ 지연)
import os
os.environ["HF_HUB_OFFLINE"] = "1"        # HuggingFace 허브 오프라인
os.environ["TRANSFORMERS_OFFLINE"] = "1"  # transformers 오프라인

# ✅ 포인트: env 설정 '후에' import·로딩 — 로컬 캐시의 BGE-M3를 씁니다.
from sentence_transformers import SentenceTransformer   # 문장 → 숫자 벡터로 바꿔주는 임베딩 모델 클래스

MODEL = "BAAI/bge-m3"                      # 임베딩 모델(다국어·1024차원)
model = SentenceTransformer(MODEL)         # 📌 로딩 약 5~10초(예시·기기마다 다름)
print("모델 로딩 완료:", MODEL)

# 차원 확인 → 1024 (vector(1024) 스키마와 정합)
# model.encode(문장)은 벡터를 돌려주고, .shape[0]으로 그 벡터가 숫자 몇 개로 이루어졌는지 확인합니다.
dim = model.encode("차원 확인용 문장").shape[0]
print("임베딩 차원 →", dim)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 34292.56it/s]


모델 로딩 완료: BAAI/bge-m3
임베딩 차원 → 1024


### 💡 힌트 (실습 1)
- **방향**: 지난 시간 시연 pgvector 시작에서 `db-pg`를 벡터 DB로 만들고 `documents` 테이블을 만든 그 코드를 참고하세요. 미니 시맨틱 검색에서 문장을 임베딩할 때 쓴 메서드도 떠올려 보세요.
- **확장**: PostgreSQL을 벡터 DB로 만드는 확장을 켜야 합니다 (`CREATE EXTENSION IF NOT EXISTS ...`).
- **컬럼 타입**: BGE-M3는 1024차원 임베딩을 만듭니다 — 그 차원 수를 담는 벡터 타입을 씁니다.
- **임베딩**: `SentenceTransformer` 모델에 문장을 넣어 벡터로 바꾸는 메서드가 있어요.
- ⭐ 임베딩 차원이 `(1024,)`로 나와야 테이블의 벡터 열과 딱 맞습니다.

---
## 🧪 실습 2. [그룹 B] 8문서 적재 & 시맨틱 검색 (예상 30분, 연계: 지난 시간 시연 미니 시맨틱 검색) ⭐

**시나리오**: 이제 한빛은행 FAQ/약관 **8문서**를 임베딩해 `documents`에 적재하고,
고객의 자연어 질문으로 **의미가 가장 가까운 문서**를 찾습니다 (시맨틱 검색).
- **질의 1**: "카드를 잃어버렸어요 어떻게 하죠?" → '체크카드 분실 재발급' 문서가 1등이어야 해요 (질문에 '분실·재발급'이란 단어가 없는데도!)
- **질의 2**: "예금자는 얼마까지 보호받나요?" → '예금자 보호법' 문서가 1등이어야 해요

**요구사항**: 8문서를 임베딩해 `documents`에 INSERT로 적재하고(임베딩 `emb`은 **float32 numpy 배열**로 변환 — `register_vector`가 `vector`로 바인딩), **코사인 거리 연산자**로 유사도를 계산·정렬하는 시맨틱 검색 함수를 작성하세요.

**✅ 확인 포인트 (순위 확정·유사도 근사)**:
- `적재 문서 수: 8`
- **질의 1 → top-1 = id 3** (체크카드 분실 재발급) · 유사도 약 **0.69** (2등: id 5 인터넷뱅킹 약 0.53)
- **질의 2 → top-1 = id 7** (예금자 보호·5천만 원) · 유사도 약 **0.73** (2등: id 1 정기예금 약 0.51)

> 💬 유사도 소수 수치가 조금 달라도 괜찮아요 — **1등 문서 id만 위와 같으면 성공!**

In [6]:
# 🧪 실습 2 — 8문서를 임베딩해 documents에 적재 → 질문과 의미가 가까운 문서를 찾는 시맨틱 검색 함수 작성
import numpy as np   # 벡터(숫자 배열) 연산에 사용

# ✅ 포인트: 각 문서를 model.encode로 임베딩 → INSERT. np.asarray(float32)로 vector에 바인딩합니다.
for content, category in DOCS:
    emb = model.encode(content)                              # 문서 → 1024차원 벡터
    conn.execute(
        # %s는 SQL 플레이스홀더 — 값을 직접 문자열로 이어붙이지 않고 안전하게 채워 넣는 방식입니다.
        "INSERT INTO documents (content, embedding, category) VALUES (%s, %s, %s)",
        (content, np.asarray(emb, dtype=np.float32), category)  # 📌 register_vector 덕에 numpy→vector 바인딩
    )
conn.commit()

cnt = conn.execute("SELECT count(*) FROM documents").fetchone()[0]
print("적재된 문서 수 →", cnt)   # 기대: 8

적재된 문서 수 → 8


In [7]:
# 질의 1: "카드를 잃어버렸어요 어떻게 하죠?"   
# ✅ 포인트: 질문도 임베딩해 <=> 코사인 검색 — '거리'가 작을수록 가까움, 1-거리 = 코사인 유사도. 
def semantic_search(query, k=2):  #꼭 k를 인자로 놓아야 하는 건 아니다 - 이 자리에 유사도 최소치 0.6 같은 걸 인자로 설정할 수도 있다
    qe = np.asarray(model.encode(query), dtype=np.float32)   # 질문 임베딩
    rows = conn.execute(
        # <=> 는 pgvector의 코사인 거리 연산자입니다. 값이 작을수록 두 벡터가 더 비슷합니다.
        # 1 - (embedding <=> %s) 로 계산하면 값이 클수록(1에 가까울수록) 더 비슷한 "유사도"가 됩니다.
        "SELECT content, category, 1 - (embedding <=> %s) AS cos_sim "
        "FROM documents ORDER BY embedding <=> %s LIMIT %s",
        (qe, qe, k)   # 같은 질문 벡터(qe)를 SELECT용·ORDER BY용으로 각각 넘겨줍니다.
    ).fetchall()
    return rows

# 질의 A: 질문에 '대출'이 없는데도 '대출' 문서를 1등으로 찾습니다(순위 확정·유사도는 근사).
print("질의 1: 카드를 잃어버렸어요 어떻게 하죠?")
for content, category, sim in semantic_search("카드를 잃어버렸어요 어떻게 하죠?"):
    print(f"  [{category}] 유사도 약 {sim:.2f}  {content}")

질의 1: 카드를 잃어버렸어요 어떻게 하죠?
  [카드] 유사도 약 0.69  체크카드를 분실한 경우 고객센터나 모바일 앱에서 즉시 재발급을 신청할 수 있습니다.
  [인터넷뱅킹] 유사도 약 0.53  인터넷뱅킹 비밀번호를 5회 잘못 입력하면 계정이 잠깁니다.


In [8]:
# 질의 2: "예금자는 얼마까지 보호받나요?"
def semantic_search(query, k=2):  #꼭 k를 인자로 놓아야 하는 건 아니다 - 이 자리에 유사도 최소치 0.6 같은 걸 인자로 설정할 수도 있다
    qe = np.asarray(model.encode(query), dtype=np.float32)   # 질문 임베딩
    rows = conn.execute(
        # <=> 는 pgvector의 코사인 거리 연산자입니다. 값이 작을수록 두 벡터가 더 비슷합니다.
        # 1 - (embedding <=> %s) 로 계산하면 값이 클수록(1에 가까울수록) 더 비슷한 "유사도"가 됩니다.
        "SELECT content, category, 1 - (embedding <=> %s) AS cos_sim "
        "FROM documents ORDER BY embedding <=> %s LIMIT %s",
        (qe, qe, k)   # 같은 질문 벡터(qe)를 SELECT용·ORDER BY용으로 각각 넘겨줍니다.
    ).fetchall()
    return rows

# 질의 A: 질문에 '대출'이 없는데도 '대출' 문서를 1등으로 찾습니다(순위 확정·유사도는 근사).
print("질의 2: 예금자는 얼마까지 보호받나요?")
for content, category, sim in semantic_search("예금자는 얼마까지 보호받나요?"):
    print(f"  [{category}] 유사도 약 {sim:.2f}  {content}")


질의 2: 예금자는 얼마까지 보호받나요?
  [약관] 유사도 약 0.73  예금자 보호법에 따라 원금과 이자를 합쳐 최대 5천만 원까지 보호됩니다.
  [예금] 유사도 약 0.51  정기예금 금리는 연 3.5% 수준이며 상품별로 다르게 적용됩니다.


### 💡 힌트 (실습 2)
- **방향**: 지난 시간 시연 미니 시맨틱 검색에서 도서관 6문서를 적재하고 검색한 흐름 그대로예요. 문서·질의만 한빛은행으로 바뀝니다.
- **적재**: `DOCS`를 순회하며 각 문장을 임베딩해 `documents`에 넣습니다 — `embedding` 열은 `vector` 타입이라 numpy 배열(float32)로 변환해서 넣어야 합니다.
- **검색**: 질문도 똑같이 임베딩한 뒤, **코사인 거리** 연산자로 가장 가까운 문서를 정렬해 가져옵니다. `1 - (거리)`가 유사도예요.
- ⭐ 코사인 거리는 **작을수록 가까워요** — `ORDER BY`에서 오름차순이 곧 '가까운 순'입니다.

---
## 🧪 실습 3. [그룹 C] ⭐필터 + 벡터 검색 결합 (예상 17분, 연계: 지난 시간 시연 메타데이터 필터 + 벡터 결합) ⭐⭐

**시나리오**: 고객이 "카드 한도를 늘리고 싶어요"라고 물었어요. 카드 문서는 **2개**(체크카드 분실·신용카드 한도)예요.
그냥 벡터 검색만 하면 다른 카테고리 문서까지 섞이니, **먼저 `카드` 카테고리로 좁힌 뒤** 그 안에서 벡터로 순위를 매깁니다.
이게 시연 메타데이터 필터 + 벡터 결합에서 배운 **"정형 필터(WHERE) + 벡터 검색을 하나의 SQL로"** — ⭐PostgreSQL이 임베딩 DB로 좋은 이유 #5의 실물이에요.

**요구사항**: 카테고리로 **먼저 좁히는 절**(정형 필터)과 질의 3에 맞는 **카테고리 값**을 넣어, 좁힌 뒤 벡터로 순위를 매기는 검색 SQL을 작성하세요.

**✅ 확인 포인트 (순위 확정·유사도 근사)**:
- **질의 3 → top-1 = id 4** (신용카드 한도) · 유사도 약 **0.70**
- 2등은 id 3 (체크카드 분실) 약 0.56 — **필터로 좁힌 '카드' 2문서 중 벡터가 '한도' 문서를 정확히 골라냄!**

> 💬 이게 오늘의 하이라이트예요 — 카테고리(정형 데이터)와 의미(벡터)를 **한 SQL로** 결합했어요.

In [9]:
# 🧪 실습 3 ⭐ — 카테고리로 좁힌 뒤(WHERE) 벡터로 순위를 매기는 필터+벡터 결합 검색
#   질의 3: "카드 한도를 늘리고 싶어요" (카드 카테고리로 좁히기)
# ✅ 포인트(⭐6축 #5): WHERE(메타 필터) + 벡터(<=>)를 '하나의 SQL'로 결합합니다.
qe = np.asarray(model.encode("카드 한도를 늘리고 싶어요"), dtype=np.float32)
rows = conn.execute(
    "SELECT content, category, 1 - (embedding <=> %s) AS cos_sim "
    "FROM documents WHERE category = %s "        # ← ① 먼저 카테고리로 좁힌 뒤
    "ORDER BY embedding <=> %s LIMIT 2",          # ← ② 벡터로 순위
    (qe, "카드", qe)
).fetchall()
print("질의 3: WHERE category='카드' + '카드 한도를 늘리고 싶어요'")
for content, category, sim in rows:
    print(f"  [{category}] 유사도 약 {sim:.2f}  {content}")


질의 3: WHERE category='카드' + '카드 한도를 늘리고 싶어요'
  [카드] 유사도 약 0.70  신용카드 이용 한도는 고객 등급과 결제 실적에 따라 조정됩니다.
  [카드] 유사도 약 0.56  체크카드를 분실한 경우 고객센터나 모바일 앱에서 즉시 재발급을 신청할 수 있습니다.


### 💡 힌트 (실습 3)
- **방향**: 지난 시간 시연 메타데이터 필터 + 벡터 결합에서 `WHERE category='시설'` + 벡터 검색으로 '주차장'을 골라낸 그 패턴이에요. 카테고리만 한빛은행 '카드'로 바뀝니다.
- **필터**: 카테고리로 먼저 좁히는 SQL 절이 필요합니다 (`ORDER BY` **앞**에 옵니다).
- **카테고리 값**: 질의가 카드 관련이니, `documents.category` 컬럼 값과 **정확히 같은 문자열**을 씁니다.
- ⭐ 실습 2의 시맨틱 검색에 필터 조건 한 줄만 더하면 됩니다.

---
## 🧪 실습 4. 🚀 (도전·선택) 자유적금 찾기 — 근소한 차이를 필터+벡터가 가른다 (예상 10분, 연계: 지난 시간 시연 메타데이터 필터 + 벡터 결합)

**시나리오**: 고객이 "매달 조금씩 넣는 상품 있나요?"라고 물었어요. 예금 문서는 **2개**(정기예금·자유적금)인데,
'매달 조금씩 넣는'은 **자유적금**(매달 자유 납입)에 더 가까워요. 하지만 두 문서의 유사도가 **아주 근소**해서,
`예금` 카테고리로 좁힌 뒤 벡터로 순위를 매겨야 자유적금이 1등으로 뽑혀요 — **필터+벡터의 힘!**

**요구사항**: 질의 4를 좁힐 **카테고리 값**을 정해, 필터+벡터 결합 검색 코드를 작성하세요.

**✅ 확인 포인트 (순위 확정·유사도 근사)**:
- **질의 4 → top-1 = id 2** (자유적금) · 유사도 약 **0.52**
- 2등은 id 1 (정기예금) 약 0.50 — **0.52 vs 0.50, 근소한 차이지만 자유적금이 1등!** (필터+벡터가 '매달 넣는' 의도를 정확히 잡아냄)

In [10]:
# 🚀 추가 도전(선택) — 질의 4를 예금 카테고리로 좁혀 검색
#   질의 4: "매달 조금씩 넣는 상품 있나요?" (예금 카테고리로 좁히기)
# 👇 아래에 직접 작성해 보세요
qe = np.asarray(model.encode("매달 조금씩 넣는 상품 있나요?"), dtype=np.float32)
rows = conn.execute(
    "SELECT content, category, 1 - (embedding <=> %s) AS cos_sim "
    "FROM documents WHERE category = %s "        # ← ① 먼저 카테고리로 좁힌 뒤
    "ORDER BY embedding <=> %s LIMIT 2",          # ← ② 벡터로 순위
    (qe, "예금", qe)
).fetchall()
print("질의 4: WHERE category='예금' + '매달 조금씩 넣는 상품 있나요?'")
for content, category, sim in rows:
    print(f"  [{category}] 유사도 약 {sim:.2f}  {content}")

질의 4: WHERE category='예금' + '매달 조금씩 넣는 상품 있나요?'
  [예금] 유사도 약 0.52  자유적금은 매달 자유롭게 납입할 수 있고 만기 시 우대 금리를 받습니다.
  [예금] 유사도 약 0.50  정기예금 금리는 연 3.5% 수준이며 상품별로 다르게 적용됩니다.


### 💡 힌트 (추가 도전)
- 실습 3과 **완전히 같은 구조**예요. 이번엔 '카드'가 아니라 **예금** 상품(정기예금·자유적금)에서 골라야 해요.
- ⭐ 필터가 없으면 다른 카테고리 문서가 섞여 순위가 흔들릴 수 있어요. 필터로 예금 2문서로 좁히면 벡터가 근소한 차이로 자유적금을 정확히 1등으로 골라내요.

---
## 🎉 수고하셨습니다! (회고 18분)

오늘 여러분이 **직접 한 것** (시연에선 도서관 문서에 했던 걸, 이번엔 한빛은행 FAQ/약관으로 내 손으로):
- **확장·스키마**: `CREATE EXTENSION vector`로 `db-pg`를 벡터 DB로 → `vector(1024)` 컬럼의 `documents` 테이블
- **임베딩 적재**: `model.encode`로 8문서를 벡터로 → `np.asarray(..., float32)`로 변환해 INSERT
- **시맨틱 검색**: `<=>`(코사인 거리)로 질문에 의미가 가까운 문서 top-1 (질의 1→체크카드 분실, 질의 2→예금자 보호)
- **⭐필터+벡터 결합**: `WHERE category='카드' ORDER BY embedding <=> 질문` — 카테고리로 좁힌 뒤 벡터로 순위 (질의 3→신용카드 한도)
- **(도전) 근소한 차이**: `WHERE category='예금'`으로 좁혀 자유적금을 정확히 (질의 4→자유적금, 0.52 vs 0.50)

### 🤔 돌아보기 질문
1. **시맨틱 검색**이 키워드 검색과 뭐가 다른가요? ('카드 분실'이란 단어가 질문에 없어도 **의미**로 찾음 — 임베딩·코사인 유사도)
2. 왜 **PostgreSQL에 임베딩**을 두면 좋을까요? (⭐6축 — 특히 업무 데이터와 **한 곳에 살고**, `WHERE`(정형 필터)+벡터를 **한 SQL로** 결합)
3. **`WHERE` 필터 + 벡터**를 결합하면 뭐가 좋은가요? (카테고리로 좁혀 **더 정확히** — 질의 3·4에서 카드/예금 2문서 중 의도에 맞는 걸 골라냄)
4. `<=>`는 왜 **작을수록** 가까운가요? (코사인 '거리'라서 — `1 - 거리`가 유사도(1에 가까울수록 비슷))
5. 유사도 수치가 아니라 **순위(top-1)** 로 채점하는 이유는? (BGE-M3는 결정적 모델이라 순위는 확정·소수 뒷자리는 부동소수점·버전에 따라 근사)
6. 이게 **RAG**에서 어떻게 쓰일까요? (검색=RAG의 **심장** — 찾은 문서를 LLM에 근거로 줌. 답 '생성'은 rag-course)

> 📌 **다음 예고** — 이어지는 노트북에서 **neo4j(그래프 DB)** 로 '친구의 친구' 같은 **관계**를 간단히 맛봅니다. neo4j는 부트캠프 후반 온톨로지·Neo4j·GraphRAG 모듈에서 본격적으로 다시 다룹니다.